# Weigel (2020), Table IV: design-aware DML estimates

This notebook re-estimates the intention-to-treat effect of randomized neighborhood assignment to the bundled property-tax campaign in Weigel (2020), *The Participation Dividend of Taxation*. It covers the seven published Table IV outcomes and respects assignment and dependence at neighborhood `a7`.

Route A uses randomization-aware cross-fitted AIPW/DML-IRM with the design propensity. Route B is a stratum-aware partially linear DML sensitivity analysis. Neither route estimates the causal effect of registration, payment, receipts, or collector visits.

## 1. Research design and data constraints

- Campaign assignment `program` is the treatment, with `stratum` fixed effects and neighborhood `a7` clustering.
- Outcome-specific opportunity rules and samples follow the replication package.
- Primary nuisance features are restricted to verified pre-campaign variables.
- Conventional Table IV replication, propensity support, data lineage, and cluster-fold feasibility are checked before DML estimation.
- The replication package identifies tax-authority name matching as confidential; this notebook neither reconstructs nor displays those records.

Design and benchmark details follow Weigel (2020), Table IV, and the accompanying replication files. Orthogonal scores and cross-fitting follow Chernozhukov et al. (2018); learner comparisons use nuisance performance rather than treatment-effect significance.

## 2. Runtime and reproducibility

This cell configures portable project paths, installs missing dependencies when needed, fixes random seeds, creates result directories, and records a runtime manifest without exposing it in notebook output.

In [ ]:
from __future__ import annotations

import hashlib
import importlib
import json
import os
import platform
import random
import subprocess
import sys
from datetime import datetime, timezone
from pathlib import Path

try:
    from google.colab import drive  # type: ignore
    IN_COLAB = True
except Exception:
    drive = None
    IN_COLAB = False

if IN_COLAB:
    drive.mount('/content/drive')

PROJECT_ROOT = Path(os.environ.get('WEIGEL_PROJECT_ROOT', Path.cwd())).expanduser().resolve()
PAPERS_DIR = PROJECT_ROOT / 'Papers'
REPLICATION_DIR = PAPERS_DIR / 'Replication_package_Weigel'
DATA_DIR = REPLICATION_DIR / 'data'
DOFILES_DIR = REPLICATION_DIR / 'dofiles'
OUTPUT_DIR = PROJECT_ROOT / 'outputs' / 'weigel_dml'
CACHE_DIR = OUTPUT_DIR / 'cache'
RESULTS_DIR = OUTPUT_DIR / 'results'
for directory in (OUTPUT_DIR, CACHE_DIR, RESULTS_DIR):
    directory.mkdir(parents=True, exist_ok=True)

MASTER_SEED = 20260623
random.seed(MASTER_SEED)
os.environ['PYTHONHASHSEED'] = str(MASTER_SEED)

REQUIRED_PACKAGES = {
    'numpy': 'numpy', 'pandas': 'pandas', 'scipy': 'scipy', 'sklearn': 'scikit-learn',
    'statsmodels': 'statsmodels', 'pyreadstat': 'pyreadstat', 'pyarrow': 'pyarrow'
}
missing = [pip_name for module, pip_name in REQUIRED_PACKAGES.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet', *missing])

import numpy as np
import pandas as pd
import scipy
import sklearn
import statsmodels

RUN_STARTED = datetime.now(timezone.utc).isoformat()
runtime_manifest = {
    'started_utc': RUN_STARTED, 'seed': MASTER_SEED, 'python': sys.version,
    'platform': platform.platform(), 'in_colab': IN_COLAB,
    'project_root': str(PROJECT_ROOT),
    'versions': {name: importlib.import_module(name).__version__ for name in REQUIRED_PACKAGES if hasattr(importlib.import_module(name), '__version__')},
}
try:
    import torch
    runtime_manifest['cuda_available'] = bool(torch.cuda.is_available())
    runtime_manifest['cuda_device'] = torch.cuda.get_device_name(0) if torch.cuda.is_available() else None
except Exception:
    runtime_manifest['cuda_available'] = False
    runtime_manifest['cuda_device'] = None

(RESULTS_DIR / 'runtime_manifest.json').write_text(json.dumps(runtime_manifest, indent=2), encoding='utf-8')
print(json.dumps(runtime_manifest, indent=2))

## 3. Targets and configuration

This cell defines the seven published targets, smoke-test and inference settings, learner menu, propensity source, and a saved analysis configuration.

In [2]:
TARGETS = pd.DataFrame([
    (1, 'townhall', 'binary', 0.045, 0.020, 1934, 252, 0.170, 0.0230),
    (2, 'evaluation', 'binary', 0.024, 0.012, 2913, 356, 0.099, 0.0580),
    (3, 'townhall_or_eval', 'binary', 0.050, 0.016, 2913, 356, 0.160, 0.0048),
    (4, 'townhall_and_eval', 'binary', 0.027, 0.009, 2913, 356, 0.035, 0.0048),
    (5, 'participation_index', 'index', 0.145, 0.043, 2913, 356, -0.077, 0.0022),
    (6, 'cost_participation_rel_w', 'continuous', 0.050, 0.017, 2913, 356, 0.110, 0.0072),
    (7, 'cost_participation2_rel_w', 'continuous', 0.071, 0.021, 2913, 356, 0.160, 0.0022),
], columns=['column', 'outcome', 'type', 'published_beta', 'published_se', 'published_n', 'published_clusters', 'control_mean', 'published_ri_p'])

# A full run uses ten repeated cluster-aware splits for stability diagnostics.
# Formal final inference uses the prespecified PRIMARY_SPLIT_SEED plus exact
# neighborhood-within-stratum randomization inference. This avoids inventing a
# nonstandard standard-error aggregation rule across repeated splits.
SMOKE_TEST = False
FORCE_RERUN = False
N_FOLDS = 2
PRIMARY_SPLIT_SEED = MASTER_SEED + 101
STABILITY_SPLIT_SEEDS = [PRIMARY_SPLIT_SEED] if SMOKE_TEST else [PRIMARY_SPLIT_SEED + j for j in range(10)]
FINAL_INFERENCE_LEARNER = 'elastic_net'
FINAL_RI_REPS = 25 if SMOKE_TEST else 5000
FINAL_RI_SEED = MASTER_SEED + 9000
FINAL_RI_OUTCOMES = TARGETS['outcome'].tolist()
FINAL_RI_ROUTES = ['A_irm_aipw', 'B_plr']
RUN_FINAL_RANDOMIZATION_INFERENCE = not SMOKE_TEST
RI_CHECKPOINT_EVERY = 10 if SMOKE_TEST else 50
DESIGN_PROPENSITY_SOURCE = 'assignment_frame'  # Used only after the propensity audit passes.
USE_VALIDATED_STATA_MERGED = False
VALIDATED_STATA_PATH = DATA_DIR / 'endline_clean_merged.dta'
PRIMARY_LEARNERS = ['ridge', 'elastic_net', 'hist_gradient_boosting']
APPROVED_EXTRA_FEATURES: list[str] = []  # Keep empty until the timing ledger is reviewed.

analysis_config = {
    'smoke_test': SMOKE_TEST, 'n_folds': N_FOLDS, 'primary_split_seed': PRIMARY_SPLIT_SEED,
    'stability_split_seeds': STABILITY_SPLIT_SEEDS, 'final_inference_learner': FINAL_INFERENCE_LEARNER,
    'final_ri_reps': FINAL_RI_REPS, 'final_ri_seed': FINAL_RI_SEED,
    'run_final_randomization_inference': RUN_FINAL_RANDOMIZATION_INFERENCE,
    'propensity_source': DESIGN_PROPENSITY_SOURCE, 'allowed_extra_features': APPROVED_EXTRA_FEATURES,
}
(RESULTS_DIR / 'analysis_config.json').write_text(json.dumps(analysis_config, indent=2), encoding='utf-8')
TARGETS.to_csv(RESULTS_DIR / 'published_table_iv_targets.csv', index=False)
display(TARGETS)

,column,outcome,type,published_beta,published_se,published_n,published_clusters,control_mean,published_ri_p
0,1,townhall,binary,0.045,0.020,1934,252,0.170,0.0230
1,2,evaluation,binary,0.024,0.012,2913,356,0.099,0.0580
2,3,townhall_or_eval,binary,0.050,0.016,2913,356,0.160,0.0048
3,4,townhall_and_eval,binary,0.027,0.009,2913,356,0.035,0.0048
4,5,participation_index,index,0.145,0.043,2913,356,-0.077,0.0022
5,6,cost_participation_rel_w,continuous,0.050,0.017,2913,356,0.110,0.0072
6,7,cost_participation2_rel_w,continuous,0.071,0.021,2913,356,0.160,0.0022


## 4. Source inventory and merge audits

This cell inventories required source files by size and hash and defines checked Stata reads, key audits, and merge auditing that never hides unmatched observations.

In [3]:
REQUIRED_DATA_FILES = [
    'endline_clean.dta', 'baseline_clean.dta', 'treatment_assignment.dta', 'flier_data.dta',
    'endline_weights.dta', 'geographic_neighborhood_level.dta', 'geographic_indiv_level.dta',
    'townhall_attendance.dta', 'submitted_evaluations.dta'
]
missing_files = [name for name in REQUIRED_DATA_FILES if not (DATA_DIR / name).exists()]
if missing_files:
    raise FileNotFoundError(f'Missing required replication files: {missing_files}. DATA_DIR={DATA_DIR}')

source_inventory = pd.DataFrame([
    {'file': name, 'path': str(DATA_DIR / name), 'bytes': (DATA_DIR / name).stat().st_size,
     'sha256': hashlib.sha256((DATA_DIR / name).read_bytes()).hexdigest()}
    for name in REQUIRED_DATA_FILES
])
source_inventory.to_csv(RESULTS_DIR / 'source_inventory.csv', index=False)
display(source_inventory[['file', 'bytes', 'sha256']])

MERGE_AUDITS: list[dict] = []

def read_stata_checked(path: Path) -> pd.DataFrame:
    frame = pd.read_stata(path, convert_categoricals=False)
    return frame.replace({8888: np.nan, 888: np.nan, 88888: np.nan, 8885: np.nan, 9999: np.nan, 7777: np.nan})

def audit_key(frame: pd.DataFrame, keys: list[str], name: str) -> dict:
    absent = [key for key in keys if key not in frame]
    if absent:
        raise KeyError(f'{name} is missing merge keys {absent}')
    return {'dataset': name, 'rows': len(frame), 'keys': ','.join(keys),
            'missing_key_rows': int(frame[keys].isna().any(axis=1).sum()),
            'duplicate_key_rows': int(frame.duplicated(keys, keep=False).sum())}

def audited_merge(left: pd.DataFrame, right: pd.DataFrame, on: list[str], name: str, validate: str | None = None) -> pd.DataFrame:
    left_audit, right_audit = audit_key(left, on, f'{name}:left'), audit_key(right, on, f'{name}:right')
    merged = left.merge(right, how='left', on=on, validate=validate, indicator=True, suffixes=('', f'__{name}'))
    MERGE_AUDITS.append({**left_audit, 'right_rows': right_audit['rows'], 'merge': name,
                         'matched_rows': int((merged['_merge'] == 'both').sum()),
                         'left_only_rows': int((merged['_merge'] == 'left_only').sum())})
    return merged.drop(columns='_merge')

,file,bytes,sha256
0,endline_clean.dta,3101380,eb697bc223f77982ddd5ddc57de935f3ec8a085e31616d...
1,baseline_clean.dta,572977,3596b2cf5d66e51b0126a76e189de3083b936f40cbcf08...
2,treatment_assignment.dta,21419,2ff843bcda3e04921361204ebf6eaf38f0d033ef79f6f0...
3,flier_data.dta,36540,e639c51f229880b4f65c2ed0263c486be60ab9056b5b55...
4,endline_weights.dta,36540,5294f2cb998b5e622716a7b38914e0bf9ab0423256136b...
5,geographic_neighborhood_level.dta,44988,ef86efdc2ac74d8392520780f32bb4b23ae00f857bf388...
6,geographic_indiv_level.dta,65757,116e500dd2376ad5cef4d4cec58e9570df07166a4534cd...
7,townhall_attendance.dta,30623,ce633b2f376c72af41d5d870c983e990fea72de6879bfa...
8,submitted_evaluations.dta,203600,4d7e197188c7e8180a6e822e44f83be581fac38573c475...


## 5. Table IV analysis dataset

The construction follows the documented endline merge order and preserves unmatched records for audit. Town-hall attendance and evaluation are coded as zero only within the documented opportunity set; otherwise they remain missing.

This cell builds the Table IV dataset from the documented source files, constructs outcomes and baseline neighborhood measures, and saves aggregate merge audits.

In [4]:
def zscore(series: pd.Series) -> pd.Series:
    sd = series.std(ddof=1)
    if not np.isfinite(sd) or sd == 0:
        raise ValueError(f'Cannot standardize a zero-variance series: {series.name}')
    return (series - series.mean()) / sd

def stata_rowtotal(frame: pd.DataFrame, columns: list[str]) -> pd.Series:
    return frame[columns].sum(axis=1, min_count=1)

def construct_baseline_neighborhoods(baseline_raw: pd.DataFrame) -> pd.DataFrame:
    required = {'a7', 'd1', 'd2', 'd2_b', 'd5', 'd7', 'd7b', 'd9', 'tax2', 'tax4', 'tax5', 'tax5b_1', 'tax5b_8888', 'tax15', 'pol3', 'pol4', 'pol5', 'f14', 'f15', 'gov1'}
    missing = sorted(required - set(baseline_raw.columns))
    if missing:
        raise KeyError(f'Baseline translation cannot be verified; missing raw fields: {missing}')
    b = baseline_raw.copy()
    b['owner'] = ((b['d7'] == 1) | (b['d7b'] == 1) | b['a7'].isin([379, 521, 667])).astype(float)
    b = b.loc[b['owner'] == 1].copy()
    b['bl_hh_size'] = b['d9'].clip(lower=1)
    b['bl_expenditure'] = b['d5']
    b['bl_prior_collector_visit'] = (b['tax5'] == 1).astype(float)
    b.loc[b['tax5b_8888'] == 1, 'bl_prior_collector_visit'] = 0
    b['bl_prior_payment'] = b['tax15'].isin([2, 4]).astype(float)
    b['bl_tax_ministry_knowledge'] = b['tax2'].isin([1, 2]).astype(float)
    b['bl_past_protest'] = (b['pol5'] == 1).astype(float)
    b['bl_vote_2011'] = (b['pol3'] == 1).astype(float)
    b['bl_party'] = (b['pol4'] == 1).astype(float)
    b['bl_political_participation'] = stata_rowtotal(b, ['bl_vote_2011', 'bl_party', 'bl_past_protest'])
    b['bl_trust_tax_ministry'] = b['f15']
    b['bl_trust_government'] = b['f14']
    b['bl_government_integrity'] = b['gov1']
    keep = [column for column in b.columns if column.startswith('bl_')]
    return b.groupby('a7', as_index=False)[keep].mean()

def construct_endline_table_iv() -> pd.DataFrame:
    endline = read_stata_checked(DATA_DIR / 'endline_clean.dta')
    assignment = read_stata_checked(DATA_DIR / 'treatment_assignment.dta')
    fliers = read_stata_checked(DATA_DIR / 'flier_data.dta')
    weights = read_stata_checked(DATA_DIR / 'endline_weights.dta')
    geo_neighborhood = read_stata_checked(DATA_DIR / 'geographic_neighborhood_level.dta')
    geo_individual = read_stata_checked(DATA_DIR / 'geographic_indiv_level.dta')
    attendance = read_stata_checked(DATA_DIR / 'townhall_attendance.dta')[['s3_code']].assign(_attended=1.0)
    evaluations = read_stata_checked(DATA_DIR / 'submitted_evaluations.dta')[['s3_code']].assign(_evaluated=1.0)
    baseline = construct_baseline_neighborhoods(read_stata_checked(DATA_DIR / 'baseline_clean.dta'))

    d = audited_merge(endline, assignment, ['a7'], 'assignment', validate='many_to_one')
    d = audited_merge(d, fliers, ['s3_code'], 'fliers', validate='one_to_one')
    d = audited_merge(d, attendance, ['s3_code'], 'attendance', validate='one_to_one')
    d = audited_merge(d, evaluations, ['s3_code'], 'evaluations', validate='one_to_one')
    d = audited_merge(d, weights, ['s3_code'], 'weights', validate='one_to_one')
    d = audited_merge(d, geo_neighborhood, ['a7'], 'geographic_neighborhood', validate='many_to_one')
    d = audited_merge(d, geo_individual, ['s3_code'], 'geographic_individual', validate='one_to_one')
    d = audited_merge(d, baseline, ['a7'], 'baseline_neighborhood', validate='many_to_one')

    d['townhall'] = np.where(d['_attended'].eq(1), 1.0, np.nan)
    d.loc[(d['_attended'].isna()) & (d['townhalls_active'].eq(1)) & (d['sample2'].eq(0)), 'townhall'] = 0.0
    d['evaluation'] = np.where(d['_evaluated'].eq(1), 1.0, np.nan)
    d.loc[(d['_evaluated'].isna()) & (d['sample2'].eq(0)), 'evaluation'] = 0.0
    d['townhall_or_eval'] = np.where(d['townhall'].notna() | d['evaluation'].notna(), ((d['townhall'] == 1) | (d['evaluation'] == 1)).astype(float), np.nan)
    d['townhall_and_eval'] = np.where(d['townhall'].notna() | d['evaluation'].notna(), ((d['townhall'] == 1) & (d['evaluation'] == 1)).astype(float), np.nan)
    d['participation_index_raw'] = stata_rowtotal(d, ['townhall', 'evaluation'])
    d['participation_index'] = zscore(d['participation_index_raw'])

    # Translate the published cost construction. These are outcomes, never DML controls.
    d['cost_participation'] = np.where(d['townhall_or_eval'].eq(1), d['transport_amount'], np.nan)
    d.loc[d['townhall_and_eval'].eq(1), 'cost_participation'] = 2 * d.loc[d['townhall_and_eval'].eq(1), 'transport_amount']
    d.loc[d['townhall_or_eval'].eq(0), 'cost_participation'] = 0.0
    inc_day = d['inc_wk'] / 7.0
    inc_day = inc_day.where(~(inc_day.isna() | inc_day.eq(0)), d['inc_mo'] / 30.0)
    upper = inc_day.quantile(0.95)
    d['inc_day_w'] = inc_day.clip(upper=upper)
    d['inc_day_poly_mean'] = d.groupby('a7')['inc_day_w'].transform('mean')
    d['cost_participation_rel_w'] = d['cost_participation'] / d['inc_day_poly_mean']
    d['inc_day_w'] = d['inc_day_w'].fillna(d['inc_day_poly_mean'])
    d['inc_hour'] = d['inc_day_w'] / 10.0
    d['cost_participation2'] = d['cost_participation']
    d.loc[d['townhall'].eq(1), 'cost_participation2'] = d.loc[d['townhall'].eq(1), 'cost_participation'] + 3 * d.loc[d['townhall'].eq(1), 'inc_hour']
    only_evaluation = d['evaluation'].eq(1) & (d['townhall'].eq(0) | d['townhall'].isna())
    d.loc[only_evaluation, 'cost_participation2'] = d.loc[only_evaluation, 'cost_participation'] + d.loc[only_evaluation, 'inc_hour']
    d.loc[d['townhall_and_eval'].eq(1), 'cost_participation2'] = d.loc[d['townhall_and_eval'].eq(1), 'cost_participation'] + 4 * d.loc[d['townhall_and_eval'].eq(1), 'inc_hour']
    d['cost_participation2_rel_w'] = d['cost_participation2'] / d['inc_day_poly_mean']

    # Exact Table IV OLS uses the endline wealth index. It remains forbidden for
    # primary DML until its timing is independently cleared in the ledger.
    d['age2'] = d['age'] ** 2
    d['roof_sum'] = d['roof'] + d['roof2'].fillna(0)
    d.loc[d['roof'].eq(6), 'roof_sum'] = 9
    d.loc[d['roof'].eq(7), 'roof_sum'] = 10
    possession_columns = [f'possessions_{j}' for j in range(1, 7)]
    d['possessions'] = stata_rowtotal(d, possession_columns)
    wealth_components = ['floor', 'roof_sum', 'walls', 'fence', 'accessible', 'elect1', 'possessions']
    standardized_components = pd.DataFrame({name: zscore(d[name]) for name in wealth_components})
    d['wealth'] = zscore(standardized_components.sum(axis=1, min_count=1))
    d.loc[d['roof'].isna(), 'wealth'] = np.nan
    income_components = ['inc_mo', 'inc_wk', 'transport', 'airtime']
    income_index = pd.DataFrame({name: zscore(d[name]) for name in income_components}).sum(axis=1, min_count=1)
    d['log_inc_for_wealth_imputation'] = np.log(zscore(income_index) + 1)
    # This matches the package's neighborhood-adjacent imputation design. A
    # failure is retained as missing rather than silently imputing from treatment.
    from sklearn.linear_model import LinearRegression
    for polygon in d.loc[d['wealth'].isna(), 'a7'].dropna().unique():
        neighborhood = d.loc[d['a7'].between(polygon - 1, polygon + 1)].copy()
        predictors = ['floor', 'log_inc_for_wealth_imputation', 'elect1', 'possessions']
        complete = neighborhood.dropna(subset=['wealth', *predictors])
        target = d['a7'].eq(polygon) & d['wealth'].isna() & d[predictors].notna().all(axis=1)
        if len(complete) >= len(predictors) + 1 and target.any():
            d.loc[target, 'wealth'] = LinearRegression().fit(complete[predictors], complete['wealth']).predict(d.loc[target, predictors])
    return d

analysis_data = construct_endline_table_iv()
pd.DataFrame(MERGE_AUDITS).to_csv(RESULTS_DIR / 'merge_audits.csv', index=False)
analysis_data.to_parquet(CACHE_DIR / 'endline_table_iv_python_build.parquet', index=False)
print(f'Python-built endline rows: {len(analysis_data):,}; neighborhoods: {analysis_data.a7.nunique():,}')
display(pd.DataFrame(MERGE_AUDITS))

Python-built endline rows: 3,536; neighborhoods: 356


,dataset,rows,keys,missing_key_rows,duplicate_key_rows,right_rows,merge,matched_rows,left_only_rows
0,assignment:left,3536,a7,0,3536,431,assignment,3536,0
1,fliers:left,3536,s3_code,0,0,3600,fliers,3536,0
2,attendance:left,3536,s3_code,0,0,379,attendance,379,3157
3,evaluations:left,3536,s3_code,0,0,320,evaluations,320,3216
4,weights:left,3536,s3_code,0,0,3600,weights,3536,0
5,geographic_neighborhood:left,3536,a7,0,3536,431,geographic_neighborhood,3536,0
6,geographic_individual:left,3536,s3_code,0,0,3547,geographic_individual,3536,0
7,baseline_neighborhood:left,3536,a7,0,3536,427,baseline_neighborhood,3536,0


## 6. Optional comparison with a validated Stata merge

When explicitly enabled, this cell compares the Python-built dataset with a validated Stata-merged artifact using row, key, assignment, and outcome checks.

In [5]:
if USE_VALIDATED_STATA_MERGED:
    if not VALIDATED_STATA_PATH.exists():
        raise FileNotFoundError(f'Validated Stata merged file requested but absent: {VALIDATED_STATA_PATH}')
    stata_merged = read_stata_checked(VALIDATED_STATA_PATH)
    compare_cols = ['s3_code', 'a7', 'program', 'stratum', *TARGETS.outcome.tolist()]
    missing_compare = [c for c in compare_cols if c not in stata_merged.columns]
    if missing_compare:
        raise KeyError(f'Validated Stata file lacks comparison columns: {missing_compare}')
    comparison = analysis_data[compare_cols].merge(stata_merged[compare_cols], on='s3_code', how='outer', indicator=True, suffixes=('_python', '_stata'))
    comparison_summary = {'python_rows': len(analysis_data), 'stata_rows': len(stata_merged), 'key_match': int((comparison['_merge'] == 'both').sum()), 'python_only': int((comparison['_merge'] == 'left_only').sum()), 'stata_only': int((comparison['_merge'] == 'right_only').sum())}
    pd.DataFrame([comparison_summary]).to_csv(RESULTS_DIR / 'python_stata_merge_comparison.csv', index=False)
    print(comparison_summary)

## 7. Feature timing ledger

Primary DML nuisance models use only features verified as pre-campaign. Endline `wealth` and `bus1` enter the conventional benchmark where required by the original specification but remain excluded from primary DML features unless their timing is independently validated.

This cell creates the executable feature-timing ledger and derives the primary nuisance feature list from variables marked as allowed.

In [6]:
feature_ledger = pd.DataFrame([
    ('a7', 'assignment', 'data_merger.do:26', 'randomization', 'neighborhood', 'identifier', 'design', 'allowed', 'Cluster key; never learned as a numeric predictor.'),
    ('stratum', 'treatment_assignment.dta', 'analysis_paper.do:51', 'pre-assignment', 'neighborhood', 'complete assignment frame', 'design', 'allowed', 'Mandatory design fixed effect / propensity stratum.'),
    ('avg_road_quality', 'geographic_neighborhood_level.dta', 'data_merger.do:146', 'pre-campaign geography pending audit', 'neighborhood', 'source-specific', 'low if verified', 'pending', 'Promote only after documenting measurement timing.'),
    ('avg_light_quality', 'geographic_neighborhood_level.dta', 'data_merger.do:146', 'pre-campaign geography pending audit', 'neighborhood', 'source-specific', 'low if verified', 'pending', 'Original OLS control; not automatically safe for DML.'),
    ('bl_hh_size', 'baseline_clean.dta', 'construct_baseline_variables.do:47-49', 'baseline', 'neighborhood mean', 'baseline survey', 'pre-treatment', 'allowed', 'Collapsed baseline household size.'),
    ('bl_expenditure', 'baseline_clean.dta', 'construct_baseline_variables.do:51-53', 'baseline', 'neighborhood mean', 'baseline survey', 'pre-treatment', 'allowed', 'Collapsed baseline expenditure.'),
    ('bl_prior_collector_visit', 'baseline_clean.dta', 'construct_baseline_variables.do:55-61', 'baseline', 'neighborhood mean', 'baseline survey', 'pre-treatment', 'allowed', 'Prior state exposure.'),
    ('bl_prior_payment', 'baseline_clean.dta', 'construct_baseline_variables.do:63-65', 'baseline', 'neighborhood mean', 'baseline survey', 'pre-treatment', 'allowed', 'Prior payment.'),
    ('bl_tax_ministry_knowledge', 'baseline_clean.dta', 'construct_baseline_variables.do:66-67', 'baseline', 'neighborhood mean', 'baseline survey', 'pre-treatment', 'allowed', 'Baseline tax knowledge.'),
    ('bl_political_participation', 'baseline_clean.dta', 'construct_baseline_variables.do:99-110', 'baseline', 'neighborhood mean', 'baseline survey', 'pre-treatment', 'allowed', 'Baseline participation summary.'),
    ('bl_trust_tax_ministry', 'baseline_clean.dta', 'construct_baseline_variables.do:69-76', 'baseline', 'neighborhood mean', 'baseline survey', 'pre-treatment', 'allowed', 'Baseline attitude; reverse coding requires source review.'),
    ('bl_trust_government', 'baseline_clean.dta', 'construct_baseline_variables.do:69-76', 'baseline', 'neighborhood mean', 'baseline survey', 'pre-treatment', 'allowed', 'Baseline attitude; reverse coding requires source review.'),
    ('wealth', 'endline_clean.dta', 'construct_endline_variables.do:84-145', 'endline', 'individual', 'imputed endline index', 'potentially post-treatment', 'pending', 'Do not use in primary DML without an approved timing argument.'),
    ('bus1', 'endline_clean.dta', 'analysis_paper.do:46', 'endline', 'individual', 'endline survey', 'potentially post-treatment', 'pending', 'Original OLS control but high risk for DML.'),
    ('registered', 'campaign/admin', 'Table III / Table VI', 'post-assignment', 'individual', 'campaign record', 'mediator', 'forbidden', 'Registration is not the ITT treatment.'),
    ('paid_receipt_union', 'payment/admin', 'Table III / Table VI', 'post-assignment', 'individual', 'campaign record', 'mediator', 'forbidden', 'Payment/receipt is not the ITT treatment.'),
    ('visited', 'midline', 'Table III', 'post-assignment', 'individual', 'survey', 'mediator', 'forbidden', 'Collector visit is a campaign component.'),
    ('double_bonus', 'collector assignment', 'analysis_paper.do:331', 'post-assignment mechanism', 'individual', 'campaign assignment', 'mechanism', 'forbidden', 'Outside primary ITT.'),
], columns=['feature', 'source_file', 'source_reference', 'measurement_period', 'level', 'missingness_rule', 'treatment_susceptibility', 'eligibility', 'explanation'])

feature_ledger.to_csv(RESULTS_DIR / 'feature_timing_ledger.csv', index=False)
display(feature_ledger)

ALLOWED_FEATURES = feature_ledger.loc[feature_ledger.eligibility.eq('allowed'), 'feature'].tolist() + APPROVED_EXTRA_FEATURES
ALLOWED_FEATURES = [f for f in ALLOWED_FEATURES if f not in {'a7', 'stratum'} and f in analysis_data.columns]
if not ALLOWED_FEATURES:
    raise RuntimeError('No verified pre-treatment nuisance features are available after the timing ledger filter.')
print('Primary allowed nuisance features:', ALLOWED_FEATURES)

,feature,source_file,source_reference,measurement_period,level,missingness_rule,treatment_susceptibility,eligibility,explanation
0,a7,assignment,data_merger.do:26,randomization,neighborhood,identifier,design,allowed,Cluster key; never learned as a numeric predic...
1,stratum,treatment_assignment.dta,analysis_paper.do:51,pre-assignment,neighborhood,complete assignment frame,design,allowed,Mandatory design fixed effect / propensity str...
2,avg_road_quality,geographic_neighborhood_level.dta,data_merger.do:146,pre-campaign geography pending audit,neighborhood,source-specific,low if verified,pending,Promote only after documenting measurement tim...
3,avg_light_quality,geographic_neighborhood_level.dta,data_merger.do:146,pre-campaign geography pending audit,neighborhood,source-specific,low if verified,pending,Original OLS control; not automatically safe f...
4,bl_hh_size,baseline_clean.dta,construct_baseline_variables.do:47-49,baseline,neighborhood mean,baseline survey,pre-treatment,allowed,Collapsed baseline household size.
5,bl_expenditure,baseline_clean.dta,construct_baseline_variables.do:51-53,baseline,neighborhood mean,baseline survey,pre-treatment,allowed,Collapsed baseline expenditure.
6,bl_prior_collector_visit,baseline_clean.dta,construct_baseline_variables.do:55-61,baseline,neighborhood mean,baseline survey,pre-treatment,allowed,Prior state exposure.
7,bl_prior_payment,baseline_clean.dta,construct_baseline_variables.do:63-65,baseline,neighborhood mean,baseline survey,pre-treatment,allowed,Prior payment.
8,bl_tax_ministry_knowledge,baseline_clean.dta,construct_baseline_variables.do:66-67,baseline,neighborhood mean,baseline survey,pre-treatment,allowed,Baseline tax knowledge.
9,bl_political_participation,baseline_clean.dta,construct_baseline_variables.do:99-110,baseline,neighborhood mean,baseline survey,pre-treatment,allowed,Baseline participation summary.


Primary allowed nuisance features: ['bl_hh_size', 'bl_expenditure', 'bl_prior_collector_visit', 'bl_prior_payment', 'bl_tax_ministry_knowledge', 'bl_political_participation', 'bl_trust_tax_ministry', 'bl_trust_government']


## 8. Locked samples and pre-estimation checks

This cell constructs outcome-specific locked samples, records critical design and sample checks, and sets `DML_UNLOCKED` only when all critical checks pass.

In [7]:
CHECKS: list[dict] = []

def record_check(name: str, passed: bool, critical: bool, detail: str) -> None:
    status = 'PASS' if passed else ('FAIL' if critical else 'WARN')
    CHECKS.append({'check': name, 'status': status, 'critical': critical, 'detail': detail})
    print(f'[{status}] {name}: {detail}')

def locked_sample(data: pd.DataFrame, outcome: str) -> pd.DataFrame:
    required = ['program', 'a7', 'stratum', outcome, *ALLOWED_FEATURES]
    missing = [column for column in required if column not in data.columns]
    if missing:
        raise KeyError(f'{outcome}: missing analysis variables {missing}')
    sample = data.loc[data[outcome].notna() & data['program'].notna() & data['a7'].notna() & data['stratum'].notna()].copy()
    return sample

def sample_summary(data: pd.DataFrame, outcome: str) -> dict:
    s = locked_sample(data, outcome)
    return {'outcome': outcome, 'n': len(s), 'clusters': s.a7.nunique(), 'treated_clusters': s.loc[s.program.eq(1), 'a7'].nunique(), 'control_clusters': s.loc[s.program.eq(0), 'a7'].nunique(), 'control_mean': float(s.loc[s.program.eq(0), outcome].mean())}

summaries = pd.DataFrame([sample_summary(analysis_data, outcome) for outcome in TARGETS.outcome]).rename(columns={'n': 'actual_n', 'clusters': 'actual_clusters', 'control_mean': 'actual_control_mean'})
comparison = TARGETS.merge(summaries, on='outcome', how='left')
comparison['n_match'] = comparison.actual_n.eq(comparison.published_n)
comparison['cluster_match'] = comparison.actual_clusters.eq(comparison.published_clusters)
comparison['control_mean_gap'] = comparison.actual_control_mean - comparison.control_mean
display(comparison)

record_check('required data files', not missing_files, True, f'missing={missing_files}')
record_check('final endline neighborhood count', analysis_data.a7.nunique() == 356, True, f'found={analysis_data.a7.nunique()} expected=356')
record_check('allowed feature ledger', all(feature_ledger.set_index('feature').loc[f, 'eligibility'] == 'allowed' for f in ALLOWED_FEATURES), True, str(ALLOWED_FEATURES))
record_check('outcome sample counts', bool(comparison.n_match.all() and comparison.cluster_match.all()), True, comparison[['outcome', 'actual_n', 'published_n', 'actual_clusters', 'published_clusters']].to_dict('records'))

CHECKS_DF = pd.DataFrame(CHECKS)
CHECKS_DF.to_csv(RESULTS_DIR / 'pre_estimation_checks_initial.csv', index=False)
DML_UNLOCKED = bool((CHECKS_DF.loc[CHECKS_DF.critical, 'status'] == 'PASS').all())
print('DML_UNLOCKED:', DML_UNLOCKED)

,column,outcome,type,published_beta,published_se,published_n,published_clusters,control_mean,published_ri_p,actual_n,actual_clusters,treated_clusters,control_clusters,actual_control_mean,n_match,cluster_match,control_mean_gap
0,1,townhall,binary,0.045,0.020,1934,252,0.170,0.0230,1934,252,160,92,0.170663,True,True,0.000663
1,2,evaluation,binary,0.024,0.012,2913,356,0.099,0.0580,2913,356,211,145,0.099174,True,True,0.000174
2,3,townhall_or_eval,binary,0.050,0.016,2913,356,0.160,0.0048,2913,356,211,145,0.164463,True,True,0.004463
3,4,townhall_and_eval,binary,0.027,0.009,2913,356,0.035,0.0048,2913,356,211,145,0.034711,True,True,-0.000289
4,5,participation_index,index,0.145,0.043,2913,356,-0.077,0.0022,2913,356,211,145,-0.076799,True,True,0.000201
5,6,cost_participation_rel_w,continuous,0.050,0.017,2913,356,0.110,0.0072,2913,356,211,145,0.110608,True,True,0.000608
6,7,cost_participation2_rel_w,continuous,0.071,0.021,2913,356,0.160,0.0022,2913,356,211,145,0.156516,True,True,-0.003484


[PASS] required data files: missing=[]
[PASS] final endline neighborhood count: found=356 expected=356
[PASS] allowed feature ledger: ['bl_hh_size', 'bl_expenditure', 'bl_prior_collector_visit', 'bl_prior_payment', 'bl_tax_ministry_knowledge', 'bl_political_participation', 'bl_trust_tax_ministry', 'bl_trust_government']
[PASS] outcome sample counts: [{'outcome': 'townhall', 'actual_n': 1934, 'published_n': 1934, 'actual_clusters': 252, 'published_clusters': 252}, {'outcome': 'evaluation', 'actual_n': 2913, 'published_n': 2913, 'actual_clusters': 356, 'published_clusters': 356}, {'outcome': 'townhall_or_eval', 'actual_n': 2913, 'published_n': 2913, 'actual_clusters': 356, 'published_clusters': 356}, {'outcome': 'townhall_and_eval', 'actual_n': 2913, 'published_n': 2913, 'actual_clusters': 356, 'published_clusters': 356}, {'outcome': 'participation_index', 'actual_n': 2913, 'published_n': 2913, 'actual_clusters': 356, 'published_clusters': 356}, {'outcome': 'cost_participation_rel_w', 'a

## 9. Conventional stratified OLS benchmark

This cell reproduces the conventional Table IV specification with stratum fixed effects and `a7`-clustered covariance, then compares coefficients and standard errors with published targets.

In [8]:
import statsmodels.formula.api as smf

ORIGINAL_OLS_CONTROLS = ['age', 'age2', 'sex', 'bus1', 'wealth', 'avg_light_quality']

def build_original_ols_controls(data: pd.DataFrame) -> pd.DataFrame:
    d = data.copy()
    if 'age2' not in d:
        d['age2'] = d['age'] ** 2
    # Exact Weigel wealth construction includes local imputations. This guard prevents false replication claims.
    if 'wealth' not in d:
        raise RuntimeError('Original OLS wealth is not yet translated exactly. Do not claim Table IV replication.')
    return d

def fit_clustered_ols(data: pd.DataFrame, outcome: str):
    d = build_original_ols_controls(data)
    needed = [outcome, 'program', 'a7', 'stratum', *ORIGINAL_OLS_CONTROLS]
    d = d.dropna(subset=needed).copy()
    formula = f'{outcome} ~ program + ' + ' + '.join(ORIGINAL_OLS_CONTROLS) + ' + C(stratum)'
    return smf.ols(formula, data=d).fit(cov_type='cluster', cov_kwds={'groups': d['a7'], 'use_correction': True}), d

ols_rows = []
for outcome in TARGETS.outcome:
    model, model_data = fit_clustered_ols(analysis_data, outcome)
    ols_rows.append({'outcome': outcome, 'ols_beta': model.params['program'], 'ols_se': model.bse['program'],
                     'ols_p': model.pvalues['program'], 'ols_n': int(model.nobs), 'ols_clusters': int(model_data.a7.nunique()),
                     'ols_control_mean': model_data.loc[model_data.program.eq(0), outcome].mean()})
OLS_RESULTS = pd.DataFrame(ols_rows)
BASELINE_COMPARISON = TARGETS.merge(OLS_RESULTS, on='outcome')
BASELINE_COMPARISON['beta_gap'] = BASELINE_COMPARISON.ols_beta - BASELINE_COMPARISON.published_beta
BASELINE_COMPARISON['se_gap'] = BASELINE_COMPARISON.ols_se - BASELINE_COMPARISON.published_se
BASELINE_COMPARISON.to_csv(RESULTS_DIR / 'baseline_table_iv_replication.csv', index=False)
display(BASELINE_COMPARISON)

BASELINE_TOLERANCE = 0.003
BASELINE_REPLICATION_PASSED = bool((BASELINE_COMPARISON.beta_gap.abs() <= BASELINE_TOLERANCE).all() and (BASELINE_COMPARISON.se_gap.abs() <= BASELINE_TOLERANCE).all())
record_check('baseline Table IV replication', BASELINE_REPLICATION_PASSED, True, f'tolerance={BASELINE_TOLERANCE}')
CHECKS_DF = pd.DataFrame(CHECKS)
CHECKS_DF.to_csv(RESULTS_DIR / 'pre_estimation_checks_after_ols.csv', index=False)
DML_UNLOCKED = bool((CHECKS_DF.loc[CHECKS_DF.critical, 'status'] == 'PASS').all())
print('DML_UNLOCKED after baseline replication:', DML_UNLOCKED)

,column,outcome,type,published_beta,published_se,published_n,published_clusters,control_mean,published_ri_p,ols_beta,ols_se,ols_p,ols_n,ols_clusters,ols_control_mean,beta_gap,se_gap
0,1,townhall,binary,0.045,0.020,1934,252,0.170,0.0230,0.045350,0.019700,0.021335,1934,252,0.170663,0.000350,-0.000300
1,2,evaluation,binary,0.024,0.012,2913,356,0.099,0.0580,0.024107,0.011959,0.043825,2913,356,0.099174,0.000107,-0.000041
2,3,townhall_or_eval,binary,0.050,0.016,2913,356,0.160,0.0048,0.049508,0.016162,0.002190,2913,356,0.164463,-0.000492,0.000162
3,4,townhall_and_eval,binary,0.027,0.009,2913,356,0.035,0.0048,0.027264,0.009461,0.003953,2913,356,0.034711,0.000264,0.000461
4,5,participation_index,index,0.145,0.043,2913,356,-0.077,0.0022,0.144563,0.042596,0.000689,2913,356,-0.076799,-0.000437,-0.000404
5,6,cost_participation_rel_w,continuous,0.050,0.017,2913,356,0.110,0.0072,0.050081,0.016530,0.002448,2913,356,0.110608,0.000081,-0.000470
6,7,cost_participation2_rel_w,continuous,0.071,0.021,2913,356,0.160,0.0022,0.070760,0.021452,0.000972,2913,356,0.156516,-0.000240,0.000452


[PASS] baseline Table IV replication: tolerance=0.003
DML_UNLOCKED after baseline replication: True


## 10. Design propensity audit

Route A uses assignment probabilities reconstructed from the randomized design within `stratum`, rather than an observational treatment classifier. The audit compares the assignment frame with each outcome-specific sample and verifies interior support.

This cell reconstructs within-stratum assignment probabilities from one record per neighborhood and audits support in every final outcome sample.

In [9]:
assignment_frame = read_stata_checked(DATA_DIR / 'treatment_assignment.dta')[['a7', 'stratum', 'program', 'nganza', 'non_residential']].drop_duplicates('a7')

def make_propensity_audit(sample: pd.DataFrame, outcome: str) -> pd.DataFrame:
    full = assignment_frame.groupby('stratum').agg(full_clusters=('a7', 'nunique'), full_treated=('program', 'sum')).reset_index()
    full['p_assignment_frame'] = full.full_treated / full.full_clusters
    # Treatment is assigned at a7. Collapse respondents before computing the
    # final-sample empirical rate; summing respondent assignments is invalid.
    final_clusters = sample[['a7', 'stratum', 'program']].drop_duplicates('a7')
    final = final_clusters.groupby('stratum').agg(final_clusters=('a7', 'nunique'), final_treated=('program', 'sum')).reset_index()
    final['p_final_sample'] = final.final_treated / final.final_clusters
    audit = full.merge(final, on='stratum', how='outer', validate='one_to_one').assign(outcome=outcome)
    audit['primary_p'] = audit['p_assignment_frame'] if DESIGN_PROPENSITY_SOURCE == 'assignment_frame' else audit['p_final_sample']
    audit['max_inverse_weight'] = np.maximum(1 / audit.primary_p, 1 / (1 - audit.primary_p))
    return audit

PROPENSITY_AUDITS = []
for outcome in TARGETS.outcome:
    audit = make_propensity_audit(locked_sample(analysis_data, outcome), outcome)
    PROPENSITY_AUDITS.append(audit)
PROPENSITY_AUDIT = pd.concat(PROPENSITY_AUDITS, ignore_index=True)
PROPENSITY_AUDIT.to_csv(RESULTS_DIR / 'design_propensity_audit.csv', index=False)
display(PROPENSITY_AUDIT)

invalid_propensity = PROPENSITY_AUDIT.primary_p.isna() | PROPENSITY_AUDIT.primary_p.le(0) | PROPENSITY_AUDIT.primary_p.ge(1)
invalid_final_rate = PROPENSITY_AUDIT.p_final_sample.isna() | PROPENSITY_AUDIT.p_final_sample.le(0) | PROPENSITY_AUDIT.p_final_sample.ge(1)
record_check('verified design propensity support', not bool(invalid_propensity.any()), True, f'source={DESIGN_PROPENSITY_SOURCE}; invalid strata={int(invalid_propensity.sum())}')
record_check('final-sample cluster-rate diagnostic', not bool(invalid_final_rate.any()), False, f'invalid strata={int(invalid_final_rate.sum())}')
CHECKS_DF = pd.DataFrame(CHECKS)
CHECKS_DF.to_csv(RESULTS_DIR / 'pre_estimation_checks_after_propensity.csv', index=False)
DML_UNLOCKED = bool((CHECKS_DF.loc[CHECKS_DF.critical, 'status'] == 'PASS').all())

,stratum,full_clusters,full_treated,p_assignment_frame,final_clusters,final_treated,p_final_sample,outcome,primary_p,max_inverse_weight
0,1.0,10,6.0,0.600000,10.0,6.0,0.600000,townhall,0.600000,2.500000
1,2.0,12,7.0,0.583333,12.0,7.0,0.583333,townhall,0.583333,2.400000
2,3.0,11,6.0,0.545455,11.0,6.0,0.545455,townhall,0.545455,2.200000
3,4.0,19,11.0,0.578947,14.0,10.0,0.714286,townhall,0.578947,2.375000
4,5.0,23,13.0,0.565217,21.0,13.0,0.619048,townhall,0.565217,2.300000
...,...,...,...,...,...,...,...,...,...,...
226,29.0,13,8.0,0.615385,13.0,8.0,0.615385,cost_participation2_rel_w,0.615385,2.600000
227,30.0,12,7.0,0.583333,12.0,7.0,0.583333,cost_participation2_rel_w,0.583333,2.400000
228,31.0,7,4.0,0.571429,7.0,4.0,0.571429,cost_participation2_rel_w,0.571429,2.333333
229,32.0,8,5.0,0.625000,8.0,5.0,0.625000,cost_participation2_rel_w,0.625000,2.666667


[PASS] verified design propensity support: source=assignment_frame; invalid strata=0
[WARN] final-sample cluster-rate diagnostic: invalid strata=44


## 11. Neighborhood- and stratum-aware folds

Folds are assigned at neighborhood `a7` and balance treatment arms within strata where support permits. This prevents respondent-level leakage across clusters and checks that every training fold can fit the required nuisance models.

This cell creates neighborhood-level folds balanced by stratum and treatment arm, saves fold audits, and verifies nuisance-training support.

In [10]:
def make_group_folds(sample: pd.DataFrame, n_folds: int, seed: int) -> pd.Series:
    groups = sample[['a7', 'stratum', 'program']].drop_duplicates('a7').copy()
    if n_folds < 2:
        raise ValueError('At least two folds are required.')
    arm_counts = groups.groupby(['stratum', 'program']).size()
    sparse_arm_strata = arm_counts.loc[arm_counts.lt(n_folds)]
    stratum_counts = groups.groupby('stratum').size()
    if stratum_counts.lt(2).any():
        raise RuntimeError(f'Cannot cross-fit a stratum represented by fewer than two final-sample clusters: {stratum_counts.loc[stratum_counts.lt(2)].to_dict()}')
    rng = np.random.default_rng(seed)
    groups['fold'] = -1
    # Allocate separately within each stratum and treatment arm. This guarantees
    # each held-out stratum is represented in each arm's training sample.
    for (_, _), block in groups.groupby(['stratum', 'program'], sort=False):
        shuffled = block.sample(frac=1, random_state=int(rng.integers(1, 2**31 - 1)))
        for position, a7 in enumerate(shuffled['a7'].tolist()):
            groups.loc[groups.a7.eq(a7), 'fold'] = position % n_folds
    if (groups.fold < 0).any():
        raise RuntimeError('Some neighborhood clusters were not assigned a fold.')
    fold_map = groups.set_index('a7')['fold']
    respondent_folds = sample['a7'].map(fold_map)
    if respondent_folds.isna().any() or sample.assign(_fold=respondent_folds).groupby('a7')['_fold'].nunique().gt(1).any():
        raise RuntimeError('A neighborhood was split across folds.')
    audit = groups.groupby('fold').agg(clusters=('a7', 'nunique'), treated=('program', 'sum')).reset_index()
    audit['control'] = audit.clusters - audit.treated
    support = groups.groupby(['stratum', 'fold', 'program']).size().rename('clusters').reset_index()
    sparse = sparse_arm_strata.rename('clusters').reset_index()
    audit.to_csv(RESULTS_DIR / f'fold_audit_seed_{seed}.csv', index=False)
    support.to_csv(RESULTS_DIR / f'fold_stratum_arm_support_seed_{seed}.csv', index=False)
    sparse.to_csv(RESULTS_DIR / f'fold_sparse_stratum_arm_seed_{seed}.csv', index=False)
    if (audit.treated.eq(0) | audit.control.eq(0)).any():
        raise RuntimeError(f'Fold support failure: {audit.to_dict("records")}')
    if not sparse.empty:
        print(f'WARN: sparse stratum-arm cells cannot appear in both folds: {sparse.to_dict("records")}. Known-propensity AIPW remains identified, but nuisance predictions partially pool these cells.')
    return respondent_folds.astype(int)

def fold_feasibility_check(data: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for outcome in TARGETS.outcome:
        sample = locked_sample(data, outcome)
        try:
            folds = make_group_folds(sample, N_FOLDS, PRIMARY_SPLIT_SEED)
            rows.append({'outcome': outcome, 'fold_feasible': True, 'detail': f'{folds.nunique()} respondent fold labels'})
        except Exception as exc:
            rows.append({'outcome': outcome, 'fold_feasible': False, 'detail': repr(exc)})
    return pd.DataFrame(rows)

FOLD_FEASIBILITY = fold_feasibility_check(analysis_data)
FOLD_FEASIBILITY.to_csv(RESULTS_DIR / 'fold_feasibility.csv', index=False)
display(FOLD_FEASIBILITY)
record_check('cluster-aware fold feasibility', bool(FOLD_FEASIBILITY.fold_feasible.all()), True, FOLD_FEASIBILITY.to_dict('records'))
CHECKS_DF = pd.DataFrame(CHECKS)
CHECKS_DF.to_csv(RESULTS_DIR / 'pre_estimation_checks_final.csv', index=False)
DML_UNLOCKED = bool((CHECKS_DF.loc[CHECKS_DF.critical, 'status'] == 'PASS').all())
print('Final DML_UNLOCKED:', DML_UNLOCKED)

WARN: sparse stratum-arm cells cannot appear in both folds: [{'stratum': 16.0, 'program': 0.0, 'clusters': 1}, {'stratum': 17.0, 'program': 0.0, 'clusters': 1}, {'stratum': 19.0, 'program': 0.0, 'clusters': 1}, {'stratum': 20.0, 'program': 0.0, 'clusters': 1}]. Known-propensity AIPW remains identified, but nuisance predictions partially pool these cells.
WARN: sparse stratum-arm cells cannot appear in both folds: [{'stratum': 16.0, 'program': 0.0, 'clusters': 1}]. Known-propensity AIPW remains identified, but nuisance predictions partially pool these cells.
WARN: sparse stratum-arm cells cannot appear in both folds: [{'stratum': 16.0, 'program': 0.0, 'clusters': 1}]. Known-propensity AIPW remains identified, but nuisance predictions partially pool these cells.
WARN: sparse stratum-arm cells cannot appear in both folds: [{'stratum': 16.0, 'program': 0.0, 'clusters': 1}]. Known-propensity AIPW remains identified, but nuisance predictions partially pool these cells.
WARN: sparse stratum-a

,outcome,fold_feasible,detail
0,townhall,True,2 respondent fold labels
1,evaluation,True,2 respondent fold labels
2,townhall_or_eval,True,2 respondent fold labels
3,townhall_and_eval,True,2 respondent fold labels
4,participation_index,True,2 respondent fold labels
5,cost_participation_rel_w,True,2 respondent fold labels
6,cost_participation2_rel_w,True,2 respondent fold labels


[PASS] cluster-aware fold feasibility: [{'outcome': 'townhall', 'fold_feasible': True, 'detail': '2 respondent fold labels'}, {'outcome': 'evaluation', 'fold_feasible': True, 'detail': '2 respondent fold labels'}, {'outcome': 'townhall_or_eval', 'fold_feasible': True, 'detail': '2 respondent fold labels'}, {'outcome': 'townhall_and_eval', 'fold_feasible': True, 'detail': '2 respondent fold labels'}, {'outcome': 'participation_index', 'fold_feasible': True, 'detail': '2 respondent fold labels'}, {'outcome': 'cost_participation_rel_w', 'fold_feasible': True, 'detail': '2 respondent fold labels'}, {'outcome': 'cost_participation2_rel_w', 'fold_feasible': True, 'detail': '2 respondent fold labels'}]
Final DML_UNLOCKED: True


## 12. Nuisance learners and tuning

This cell defines modest ridge, elastic-net, and histogram-boosting nuisance learners with preprocessing and tuning contained within training folds.

In [11]:
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import ElasticNet, Ridge
from sklearn.metrics import brier_score_loss, mean_squared_error, log_loss
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

def learner_factory(name: str, is_binary: bool, seed: int):
    numeric = Pipeline([('imputer', SimpleImputer(strategy='median')), ('scale', StandardScaler())])
    categorical = Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))])
    preprocessor = ColumnTransformer([('numeric', numeric, ALLOWED_FEATURES), ('stratum', categorical, ['stratum'])], remainder='drop')
    if name == 'ridge':
        model = Ridge(alpha=5.0, random_state=seed)
    elif name == 'elastic_net':
        model = ElasticNet(alpha=0.03, l1_ratio=0.5, max_iter=20000, random_state=seed)
    elif name == 'hist_gradient_boosting':
        # Trees require dense transformed features. Keep depth low to control variance.
        model = HistGradientBoostingRegressor(max_depth=3, learning_rate=0.05, max_iter=150, l2_regularization=1.0, random_state=seed)
    else:
        raise ValueError(f'Unknown learner: {name}')
    return Pipeline([('preprocess', preprocessor), ('model', model)])

def nuisance_loss(y_true: np.ndarray, prediction: np.ndarray, outcome_type: str) -> dict:
    if outcome_type == 'binary':
        clipped = np.clip(prediction, 1e-6, 1 - 1e-6)
        return {'mse': float(mean_squared_error(y_true, prediction)), 'brier': float(brier_score_loss(y_true, clipped)), 'log_loss': float(log_loss(y_true, clipped, labels=[0, 1]))}
    return {'mse': float(mean_squared_error(y_true, prediction)), 'brier': np.nan, 'log_loss': np.nan}

## 13. Cluster-robust inference

These utilities aggregate influence contributions by neighborhood and compute finite-cluster-corrected uncertainty and p-values for AIPW and PLR scores.

In [12]:
from scipy import stats

def clustered_mean_inference(influence: np.ndarray, clusters: pd.Series) -> dict:
    frame = pd.DataFrame({'influence': np.asarray(influence, dtype=float), 'cluster': np.asarray(clusters)})
    grouped = frame.groupby('cluster', sort=False)['influence'].sum()
    n, g = len(frame), len(grouped)
    if g < 2:
        raise ValueError('At least two clusters are required for clustered inference.')
    variance = (g / (g - 1)) * np.sum(grouped.to_numpy() ** 2) / (n ** 2)
    se = float(np.sqrt(variance))
    return {'se': se, 'clusters': g, 'cluster_influence': grouped}

def clustered_plr_inference(d_tilde: np.ndarray, residual: np.ndarray, clusters: pd.Series) -> dict:
    d_tilde = np.asarray(d_tilde, dtype=float)
    residual = np.asarray(residual, dtype=float)
    frame = pd.DataFrame({'score': d_tilde * residual, 'cluster': np.asarray(clusters)})
    scores = frame.groupby('cluster', sort=False)['score'].sum()
    n, g = len(frame), len(scores)
    denominator = float(np.sum(d_tilde ** 2))
    if denominator <= 1e-10:
        raise ValueError('Near-zero residual treatment variation in PLR final stage.')
    variance = (g / (g - 1)) * np.sum(scores.to_numpy() ** 2) / (denominator ** 2)
    return {'se': float(np.sqrt(variance)), 'clusters': g, 'cluster_score': scores, 'denominator': denominator}

def infer_pvalue(beta: float, se: float, clusters: int) -> tuple[float, float, float]:
    df = clusters - 1
    critical = stats.t.ppf(0.975, df)
    t_value = beta / se
    return float(2 * stats.t.sf(abs(t_value), df)), float(beta - critical * se), float(beta + critical * se)

## 14. Route A: randomization-aware AIPW/DML-IRM

Route A estimates treatment-specific out-of-fold outcome regressions and combines them with the verified within-stratum design propensity. Neighborhood-summed influence contributions determine uncertainty.

This function implements Route A with treatment-specific held-out outcome predictions, the verified design propensity, and neighborhood-clustered influence inference.

In [13]:
def design_propensity_for_sample(sample: pd.DataFrame, outcome: str) -> pd.Series:
    audit = PROPENSITY_AUDIT.loc[PROPENSITY_AUDIT.outcome.eq(outcome), ['stratum', 'primary_p']]
    mapping = audit.drop_duplicates('stratum').set_index('stratum')['primary_p']
    p = sample['stratum'].map(mapping)
    if p.isna().any() or p.le(0).any() or p.ge(1).any():
        raise RuntimeError(f'{outcome}: unverified or unsupported design propensity.')
    return p.astype(float)

def run_route_a_irm(sample: pd.DataFrame, outcome: str, outcome_type: str, learner_name: str, split_seed: int) -> tuple[dict, pd.DataFrame]:
    work = sample.reset_index(drop=True).copy()
    work['_fold'] = make_group_folds(work, N_FOLDS, split_seed).to_numpy()
    work['_p'] = design_propensity_for_sample(work, outcome)
    mu0, mu1 = np.full(len(work), np.nan), np.full(len(work), np.nan)
    losses: list[dict] = []
    for fold in sorted(work._fold.unique()):
        train, test = work.loc[work._fold.ne(fold)], work.loc[work._fold.eq(fold)]
        for arm, target in ((0, mu0), (1, mu1)):
            train_arm = train.loc[train.program.eq(arm)]
            if train_arm.a7.nunique() < 5:
                raise RuntimeError(f'{outcome}, fold={fold}, arm={arm}: insufficient training clusters.')
            learner = learner_factory(learner_name, outcome_type == 'binary', split_seed + fold * 10 + arm)
            learner.fit(train_arm[[*ALLOWED_FEATURES, 'stratum']], train_arm[outcome])
            prediction = np.asarray(learner.predict(test[[*ALLOWED_FEATURES, 'stratum']]), dtype=float)
            if outcome_type == 'binary':
                prediction = np.clip(prediction, 0.0, 1.0)
            target[test.index.to_numpy()] = prediction
            loss = nuisance_loss(test[outcome].to_numpy(), prediction, outcome_type)
            losses.append({'fold': fold, 'arm_model': arm, **loss})
    if np.isnan(mu0).any() or np.isnan(mu1).any():
        raise RuntimeError(f'{outcome}: missing out-of-fold Route A nuisance predictions.')
    p = work['_p'].to_numpy()
    d, y = work.program.to_numpy(float), work[outcome].to_numpy(float)
    clipped_p = np.clip(p, 1e-6, 1 - 1e-6)
    if not np.allclose(p, clipped_p):
        raise RuntimeError('Design propensity required numerical clipping; resolve the propensity audit instead.')
    score = mu1 - mu0 + d * (y - mu1) / p - (1 - d) * (y - mu0) / (1 - p)
    beta = float(np.mean(score))
    inference = clustered_mean_inference(score - beta, work.a7)
    pvalue, ci_low, ci_high = infer_pvalue(beta, inference['se'], inference['clusters'])
    result = {'route': 'A_irm_aipw', 'outcome': outcome, 'learner': learner_name, 'split_seed': split_seed,
              'estimate': beta, 'se_cluster': inference['se'], 'p_cluster': pvalue, 'ci_low': ci_low, 'ci_high': ci_high,
              'n': len(work), 'clusters': inference['clusters'], 'propensity_source': DESIGN_PROPENSITY_SOURCE,
              'y_mse': float(np.mean([x['mse'] for x in losses])), 'max_inverse_weight': float(max((1 / p).max(), (1 / (1-p)).max()))}
    artifacts = work[['s3_code', 'a7', 'stratum', 'program', outcome, '_fold', '_p']].copy()
    artifacts['mu0_oof'], artifacts['mu1_oof'], artifacts['aipw_score'] = mu0, mu1, score
    return result, artifacts

## 15. Route B: stratum-aware PLR-DML

Route B residualizes the outcome and treatment using fold-specific nuisance models with stratum retained as a design component. It is a secondary sensitivity analysis, with `a7`-clustered final-stage inference.

This function implements Route B with fold-aware stratum adjustment, design-propensity treatment residualization, and an `a7`-clustered residual-on-residual final stage.

In [14]:
def run_route_b_plr(sample: pd.DataFrame, outcome: str, outcome_type: str, learner_name: str, split_seed: int) -> tuple[dict, pd.DataFrame]:
    work = sample.reset_index(drop=True).copy()
    work['_fold'] = make_group_folds(work, N_FOLDS, split_seed).to_numpy()
    work['_m'] = design_propensity_for_sample(work, outcome)
    ghat = np.full(len(work), np.nan)
    losses: list[dict] = []
    for fold in sorted(work._fold.unique()):
        train, test = work.loc[work._fold.ne(fold)], work.loc[work._fold.eq(fold)]
        learner = learner_factory(learner_name, outcome_type == 'binary', split_seed + fold)
        learner.fit(train[[*ALLOWED_FEATURES, 'stratum']], train[outcome])
        prediction = np.asarray(learner.predict(test[[*ALLOWED_FEATURES, 'stratum']]), dtype=float)
        if outcome_type == 'binary':
            prediction = np.clip(prediction, 0.0, 1.0)
        ghat[test.index.to_numpy()] = prediction
        losses.append({'fold': fold, **nuisance_loss(test[outcome].to_numpy(), prediction, outcome_type)})
    if np.isnan(ghat).any():
        raise RuntimeError(f'{outcome}: missing out-of-fold Route B outcome nuisance predictions.')
    y_tilde = work[outcome].to_numpy(float) - ghat
    d_tilde = work.program.to_numpy(float) - work['_m'].to_numpy(float)
    denominator = float(np.sum(d_tilde ** 2))
    if denominator <= 1e-10:
        raise RuntimeError(f'{outcome}: near-zero residual assignment variation.')
    beta = float(np.sum(d_tilde * y_tilde) / denominator)
    final_residual = y_tilde - beta * d_tilde
    inference = clustered_plr_inference(d_tilde, final_residual, work.a7)
    pvalue, ci_low, ci_high = infer_pvalue(beta, inference['se'], inference['clusters'])
    result = {'route': 'B_plr', 'outcome': outcome, 'learner': learner_name, 'split_seed': split_seed,
              'estimate': beta, 'se_cluster': inference['se'], 'p_cluster': pvalue, 'ci_low': ci_low, 'ci_high': ci_high,
              'n': len(work), 'clusters': inference['clusters'], 'propensity_source': DESIGN_PROPENSITY_SOURCE,
              'y_mse': float(np.mean([x['mse'] for x in losses])), 'residual_treatment_ss': denominator}
    artifacts = work[['s3_code', 'a7', 'stratum', 'program', outcome, '_fold', '_m']].copy()
    artifacts['g_hat_oof'], artifacts['y_tilde'], artifacts['d_tilde'], artifacts['final_residual'] = ghat, y_tilde, d_tilde, final_residual
    return result, artifacts

## 16. Design-consistent randomization inference

Formal DML p-values use the prespecified learner and primary cluster fold with neighborhood assignments permuted within strata. Full runs use 5,000 draws and refit the complete estimator; repeated split estimates are reported separately as stability diagnostics.

These functions permute neighborhood treatment within strata, run conventional and DML randomization inference, checkpoint draws, and assemble final inference tables.

In [15]:
def permute_program_by_stratum(sample: pd.DataFrame, seed: int) -> pd.Series:
    clusters = sample[['a7', 'stratum', 'program']].drop_duplicates('a7').reset_index(drop=True)
    permuted = clusters['program'].to_numpy(dtype=float).copy()
    rng = np.random.default_rng(seed)
    for _, positions in clusters.groupby('stratum', sort=False).indices.items():
        positions = np.asarray(positions, dtype=int)
        permuted[positions] = rng.permutation(permuted[positions])
    permuted_clusters = clusters.assign(program_permuted=permuted)
    original_counts = clusters.groupby('stratum')['program'].sum()
    permuted_counts = permuted_clusters.groupby('stratum')['program_permuted'].sum()
    permuted_counts = permuted_counts.reindex(original_counts.index)
    if not np.allclose(
        original_counts.to_numpy(dtype=float),
        permuted_counts.to_numpy(dtype=float),
        rtol=0,
        atol=0,
    ):
        raise RuntimeError('Permutation failed to preserve within-stratum treatment counts.')
    return sample['a7'].map(permuted_clusters.set_index('a7')['program_permuted']).astype(float)

def ols_randomization_pvalue(sample: pd.DataFrame, outcome: str, reps: int, seed: int) -> dict:
    observed_model, observed_data = fit_clustered_ols(sample, outcome)
    observed = float(observed_model.params['program'])
    draws = []
    formula = f'{outcome} ~ program + ' + ' + '.join(ORIGINAL_OLS_CONTROLS) + ' + C(stratum)'
    for rep in range(reps):
        draw = observed_data.copy()
        draw['program'] = permute_program_by_stratum(draw, seed + rep).to_numpy()
        draws.append(float(smf.ols(formula, data=draw).fit().params['program']))
    draws = np.asarray(draws)
    return {'outcome': outcome, 'ols_ri_reps': reps, 'ols_ri_p': float((1 + np.sum(np.abs(draws) >= abs(observed))) / (reps + 1)), 'ols_observed': observed}

def _final_estimator(route: str):
    if route == 'A_irm_aipw':
        return run_route_a_irm
    if route == 'B_plr':
        return run_route_b_plr
    raise ValueError(f'Unknown final-inference route: {route}')

def _ri_paths(route: str, outcome: str) -> tuple[Path, Path]:
    stem = f'ri_{route}_{outcome}_{FINAL_INFERENCE_LEARNER}_seed{FINAL_RI_SEED}_reps{FINAL_RI_REPS}'
    return RESULTS_DIR / f'{stem}_draws.parquet', RESULTS_DIR / f'{stem}_summary.json'

def exact_dml_randomization_inference(sample: pd.DataFrame, outcome: str, outcome_type: str, route: str) -> tuple[dict, pd.DataFrame]:
    estimator = _final_estimator(route)
    observed_result, observed_artifact = estimator(sample, outcome, outcome_type, FINAL_INFERENCE_LEARNER, PRIMARY_SPLIT_SEED)
    draws_path, summary_path = _ri_paths(route, outcome)
    if draws_path.exists() and not FORCE_RERUN:
        draws_frame = pd.read_parquet(draws_path)
    else:
        draws_frame = pd.DataFrame(columns=['replication', 'estimate'])
    completed = set(draws_frame.get('replication', pd.Series(dtype=int)).astype(int).tolist())
    new_draws = []
    for rep in range(FINAL_RI_REPS):
        if rep in completed:
            continue
        permuted_sample = sample.copy()
        permuted_sample['program'] = permute_program_by_stratum(permuted_sample, FINAL_RI_SEED + rep).to_numpy()
        permuted_result, _ = estimator(permuted_sample, outcome, outcome_type, FINAL_INFERENCE_LEARNER, PRIMARY_SPLIT_SEED)
        new_draws.append({'replication': rep, 'estimate': permuted_result['estimate']})
        if len(new_draws) >= RI_CHECKPOINT_EVERY:
            draws_frame = pd.concat([draws_frame, pd.DataFrame(new_draws)], ignore_index=True)
            draws_frame.to_parquet(draws_path, index=False)
            print(f'RI checkpoint: route={route}, outcome={outcome}, completed={len(draws_frame)}/{FINAL_RI_REPS}')
            new_draws = []
    if new_draws:
        draws_frame = pd.concat([draws_frame, pd.DataFrame(new_draws)], ignore_index=True)
        draws_frame.to_parquet(draws_path, index=False)
    if len(draws_frame) != FINAL_RI_REPS:
        raise RuntimeError(f'RI incomplete for {route}/{outcome}: {len(draws_frame)} of {FINAL_RI_REPS} draws.')
    ri_p = float((1 + np.sum(np.abs(draws_frame['estimate'].to_numpy()) >= abs(observed_result['estimate']))) / (FINAL_RI_REPS + 1))
    final_result = {**observed_result, 'final_inference': True, 'ri_reps': FINAL_RI_REPS, 'ri_p_exact': ri_p,
                    'ri_draws_path': str(draws_path), 'final_learner': FINAL_INFERENCE_LEARNER}
    summary_path.write_text(json.dumps({key: value for key, value in final_result.items() if key not in {'cluster_influence', 'cluster_score'}}, indent=2, default=str), encoding='utf-8')
    return final_result, observed_artifact

def run_final_inference(data: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    if not DML_UNLOCKED:
        raise RuntimeError('Final inference is locked until all critical checks pass.')
    point_rows, ri_rows = [], []
    for target in TARGETS.itertuples(index=False):
        sample = locked_sample(data, target.outcome)
        for route in FINAL_RI_ROUTES:
            estimator = _final_estimator(route)
            if RUN_FINAL_RANDOMIZATION_INFERENCE:
                result, artifact = exact_dml_randomization_inference(sample, target.outcome, target.type, route)
            else:
                result, artifact = estimator(sample, target.outcome, target.type, FINAL_INFERENCE_LEARNER, PRIMARY_SPLIT_SEED)
                result.update({'final_inference': False, 'ri_reps': 0, 'ri_p_exact': np.nan, 'final_learner': FINAL_INFERENCE_LEARNER})
            result['run_id'] = configuration_hash({'route': route, 'outcome': target.outcome, 'learner': FINAL_INFERENCE_LEARNER, 'split': PRIMARY_SPLIT_SEED, 'ri_reps': result['ri_reps']})
            artifact.to_parquet(RESULTS_DIR / f"final_oof_{result['run_id']}.parquet", index=False)
            point_rows.append(result)
            ri_rows.append({'route': route, 'outcome': target.outcome, 'ri_p_exact': result['ri_p_exact'], 'ri_reps': result['ri_reps'], 'run_id': result['run_id']})
            print(f"Final inference complete: {route} / {target.outcome}; RI p={result['ri_p_exact']}")
    points, ri = pd.DataFrame(point_rows), pd.DataFrame(ri_rows)
    points.to_csv(RESULTS_DIR / 'final_dml_inference.csv', index=False)
    ri.to_csv(RESULTS_DIR / 'final_dml_randomization_inference.csv', index=False)
    return points, ri

def run_ols_randomization_inference(data: pd.DataFrame) -> pd.DataFrame:
    if not RUN_FINAL_RANDOMIZATION_INFERENCE:
        return pd.DataFrame(columns=['outcome', 'ols_ri_reps', 'ols_ri_p', 'ols_observed'])
    rows = []
    for target in TARGETS.itertuples(index=False):
        print(f'OLS RI: {target.outcome} ({FINAL_RI_REPS} repetitions)')
        rows.append(ols_randomization_pvalue(data, target.outcome, FINAL_RI_REPS, FINAL_RI_SEED))
    frame = pd.DataFrame(rows)
    frame.to_csv(RESULTS_DIR / 'final_ols_randomization_inference.csv', index=False)
    return frame

This cell defines configuration hashes, append-only run logging, stable run identifiers, and the controller for every route–outcome–learner–split combination.

In [16]:
def configuration_hash(payload: dict) -> str:
    return hashlib.sha256(
        json.dumps(payload, sort_keys=True, default=str).encode("utf-8")
    ).hexdigest()[:16]


def append_result(row: dict) -> None:
    path = RESULTS_DIR / "dml_run_log.csv"
    pd.DataFrame([row]).to_csv(
        path,
        mode="a",
        header=not path.exists(),
        index=False,
    )


def run_id(route: str, outcome: str, learner: str, split_seed: int) -> str:
    payload = {
        "route": route,
        "outcome": outcome,
        "learner": learner,
        "split_seed": split_seed,
        "features": ALLOWED_FEATURES,
        "propensity_source": DESIGN_PROPENSITY_SOURCE,
        "targets": TARGETS.to_dict("records"),
    }
    return configuration_hash(payload)


def run_all_dml(data: pd.DataFrame) -> pd.DataFrame:
    if not DML_UNLOCKED:
        raise RuntimeError(
            "DML is locked. Resolve every critical pre-estimation failure "
            "before fitting any learner."
        )

    rows = []

    for target in TARGETS.itertuples(index=False):
        sample = locked_sample(data, target.outcome)

        for split_seed in STABILITY_SPLIT_SEEDS:
            for learner in PRIMARY_LEARNERS:
                for route_name, estimator in [
                    ("A_irm_aipw", run_route_a_irm),
                    ("B_plr", run_route_b_plr),
                ]:
                    identifier = run_id(
                        route_name,
                        target.outcome,
                        learner,
                        split_seed,
                    )
                    artifact_path = RESULTS_DIR / f"oof_{identifier}.parquet"

                    try:
                        result, artifacts = estimator(
                            sample,
                            target.outcome,
                            target.type,
                            learner,
                            split_seed,
                        )
                        result.update(
                            {
                                "run_id": identifier,
                                "status": "success",
                                "timestamp_utc": datetime.now(
                                    timezone.utc
                                ).isoformat(),
                            }
                        )
                        artifacts.to_parquet(artifact_path, index=False)

                    except Exception as exc:
                        result = {
                            "route": route_name,
                            "outcome": target.outcome,
                            "learner": learner,
                            "split_seed": split_seed,
                            "run_id": identifier,
                            "status": "failed",
                            "error": repr(exc),
                            "timestamp_utc": datetime.now(
                                timezone.utc
                            ).isoformat(),
                        }

                    append_result(result)
                    rows.append(result)
                    print(
                        result["status"],
                        route_name,
                        target.outcome,
                        learner,
                        split_seed,
                    )

    return pd.DataFrame(rows)

## 17. Estimation controller and restart safety

This cell runs the repeated-split stability batch and the prespecified final DML and OLS randomization-inference analyses. Its transient progress output is intentionally omitted from the public notebook.

In [ ]:
DML_RUNS = run_all_dml(analysis_data)
DML_RUNS.to_parquet(RESULTS_DIR / 'dml_runs_stability_diagnostics.parquet', index=False)

# The repeated-split run above is a stability diagnostic. Final inference below
# uses a prespecified learner and split, and (in a full run) exact stratified
# neighborhood-level randomization inference with full nuisance refitting.
FINAL_DML_RESULTS, FINAL_DML_RI = run_final_inference(analysis_data)
FINAL_OLS_RI = run_ols_randomization_inference(analysis_data)

display(DML_RUNS)
display(FINAL_DML_RESULTS)
display(FINAL_OLS_RI)

## 18. Results, diagnostics, and multiple testing

This cell produces the final OLS–Route A–Route B comparison, split-stability diagnostics, and Sankoh adjustments for the town-hall and evaluation outcome family.

In [18]:
stability_summary = (DML_RUNS.loc[DML_RUNS.status.eq('success')]
                     .groupby(['route', 'outcome', 'learner'], as_index=False)
                     .agg(split_count=('estimate', 'size'), estimate_mean=('estimate', 'mean'), estimate_sd_across_splits=('estimate', 'std'),
                          estimate_min=('estimate', 'min'), estimate_max=('estimate', 'max'), mean_nuisance_mse=('y_mse', 'mean')))
stability_summary.to_csv(RESULTS_DIR / 'dml_split_stability_summary.csv', index=False)

route_a_final = FINAL_DML_RESULTS.loc[FINAL_DML_RESULTS.route.eq('A_irm_aipw')].add_prefix('route_a_')
route_b_final = FINAL_DML_RESULTS.loc[FINAL_DML_RESULTS.route.eq('B_plr')].add_prefix('route_b_')
paper_style = TARGETS.merge(OLS_RESULTS, on='outcome', how='left').merge(
    FINAL_OLS_RI.rename(columns={'ols_ri_p': 'replicated_ols_ri_p'}), on='outcome', how='left'
).merge(route_a_final, left_on='outcome', right_on='route_a_outcome', how='left').merge(
    route_b_final, left_on='outcome', right_on='route_b_outcome', how='left'
)
paper_style.to_csv(RESULTS_DIR / 'table_iv_final_ols_route_a_route_b_comparison.csv', index=False)
display(paper_style)
display(stability_summary)

diagnostic_tables = {
    'checks': CHECKS_DF, 'fold_feasibility': FOLD_FEASIBILITY,
    'propensity_audit': PROPENSITY_AUDIT, 'feature_ledger': feature_ledger,
    'stability_runs': DML_RUNS, 'stability_summary': stability_summary,
    'final_dml': FINAL_DML_RESULTS, 'final_ols_ri': FINAL_OLS_RI,
}
for name, frame in diagnostic_tables.items():
    frame.to_csv(RESULTS_DIR / f'diagnostic_{name}.csv', index=False)

def sankoh_adjustment(p_values: pd.Series, rho: float) -> pd.Series:
    g = 2 ** (1 - rho)
    return 1 - (1 - p_values) ** g

rho = float(locked_sample(analysis_data, 'evaluation')[['townhall', 'evaluation']].corr().iloc[0, 1])
SANKOH_OLS = FINAL_OLS_RI.loc[FINAL_OLS_RI.outcome.isin(['townhall', 'evaluation'])].copy()
SANKOH_OLS['rho'] = rho
SANKOH_OLS['p_sankoh'] = sankoh_adjustment(SANKOH_OLS['ols_ri_p'], rho)
SANKOH_DML = FINAL_DML_RESULTS.loc[FINAL_DML_RESULTS.outcome.isin(['townhall', 'evaluation']), ['route', 'outcome', 'ri_p_exact', 'ri_reps']].copy()
SANKOH_DML['rho'] = rho
SANKOH_DML['p_sankoh'] = sankoh_adjustment(SANKOH_DML['ri_p_exact'], rho)
SANKOH_OLS.to_csv(RESULTS_DIR / 'sankoh_ols_randomization_adjustment.csv', index=False)
SANKOH_DML.to_csv(RESULTS_DIR / 'sankoh_dml_randomization_adjustment.csv', index=False)
display(SANKOH_OLS)
display(SANKOH_DML)

,route,outcome,learner,split_count,estimate_mean,estimate_sd_across_splits,estimate_min,estimate_max,mean_nuisance_mse
0,A_irm_aipw,cost_participation2_rel_w,elastic_net,10,0.069725,0.003566,0.065472,0.076652,0.280659
1,A_irm_aipw,cost_participation2_rel_w,hist_gradient_boosting,10,0.050068,0.004941,0.039957,0.055761,0.292958
2,A_irm_aipw,cost_participation2_rel_w,ridge,10,0.073475,0.005303,0.063135,0.081522,0.296963
3,A_irm_aipw,cost_participation_rel_w,elastic_net,10,0.049079,0.002431,0.046038,0.054868,0.166069
4,A_irm_aipw,cost_participation_rel_w,hist_gradient_boosting,10,0.035814,0.003610,0.028800,0.040069,0.174759
5,A_irm_aipw,cost_participation_rel_w,ridge,10,0.053469,0.004102,0.045230,0.058991,0.175388
6,A_irm_aipw,evaluation,elastic_net,10,0.018915,0.000892,0.017222,0.020078,0.098395
7,A_irm_aipw,evaluation,hist_gradient_boosting,10,0.015796,0.004109,0.008029,0.020730,0.101667
8,A_irm_aipw,evaluation,ridge,10,0.016387,0.002298,0.011668,0.019608,0.102876
9,A_irm_aipw,participation_index,elastic_net,10,0.133617,0.004529,0.127204,0.140830,1.018108


,outcome,ols_ri_reps,ols_ri_p,ols_observed,rho,p_sankoh
0,townhall,5000,0.028594,0.045350,0.397525,0.043092
1,evaluation,5000,0.056389,0.024107,0.397525,0.084353


,route,outcome,ri_p_exact,ri_reps,rho,p_sankoh
0,A_irm_aipw,townhall,0.031594,5000,0.397525,0.047575
1,B_plr,townhall,0.023395,5000,0.397525,0.035305
2,A_irm_aipw,evaluation,0.164367,5000,0.397525,0.238632
3,B_plr,evaluation,0.131174,5000,0.397525,0.192243
